#### Since the Minimal Preprocessing approach did not work, here in this Notebook we'll do targeted error analysis by inspecting the data, if they are correctly labelled or not in the first place

In [1]:
import pandas as pd

In [2]:
# Loading the Dataset, and observing the Positive class
df = pd.read_csv('indian_liver_patient.csv')

In [3]:
df[df["Dataset"]==1].head()

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182,14,20,6.8,3.4,1.00,1
4,72,Male,3.9,2.0,195,27,59,7.3,2.4,0.40,1


In [4]:
# Negatative Class
df[df["Dataset"] == 2].head()

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
8,17,Male,0.9,0.3,202,22,19,7.4,4.1,1.20,2
12,64,Male,0.9,0.3,310,61,58,7.0,3.4,0.90,2
15,25,Male,0.6,0.1,183,91,53,5.5,2.3,0.70,2
17,33,Male,1.6,0.5,165,15,23,7.3,3.5,0.92,2
24,63,Male,0.9,0.2,194,52,45,6.0,3.9,1.85,2


In [5]:
# Converting Dataset column to 0/1 (1 = disease, 0 = no disease)
df["Dataset"] = df["Dataset"].map({1:1, 2:0})

In [6]:
# Dropping the duplicated rows
df.drop_duplicates(inplace=True)

In [7]:
df["Dataset"].value_counts()

Dataset
1    406
0    164
Name: count, dtype: int64

##  Data Understanding

The normal range of Total Bilirubin is typically:
 * Total Bilirubin: 0.3 to 1.2 mg/dL (milligrams per deciliter)

In [8]:
# Inspecting Total Bilirubin
total_bilirubin = df[(df["Total_Bilirubin"]<0.3) | (df["Total_Bilirubin"]>1.2)]
total_bilirubin

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490,60,68,7.0,3.3,0.89,1
4,72,Male,3.9,2.0,195,27,59,7.3,2.4,0.40,1
5,46,Male,1.8,0.7,208,19,14,7.6,4.4,1.30,1
11,72,Male,2.7,1.3,260,31,56,7.4,3.0,0.60,1
...,...,...,...,...,...,...,...,...,...,...,...
574,32,Male,12.1,6.0,515,48,92,6.6,2.4,0.50,1
575,32,Male,25.0,13.7,560,41,88,7.9,2.5,2.50,1
576,32,Male,15.0,8.2,289,58,80,5.3,2.2,0.70,1
577,32,Male,12.7,8.4,190,28,47,5.4,2.6,0.90,1


Above 1.2 mg/dL is considered elevated. [Mostly an Alarm]

In [9]:
total_bilirubin["Dataset"].value_counts()

Dataset
1    212
0     33
Name: count, dtype: int64

There are 33 Records that are Normal despite having elevated Total_Bilirubin

In [10]:
tb_negative = total_bilirubin[total_bilirubin["Dataset"]==0]

<ul>
    <li>One cannot say these 33 rows are mislabeled just because Total_Bilirubin is abnormal.</li>
    <li>Elevated bilirubin ≠ guaranteed liver disease.</li>
    <li>Labels may be correct due to clinical context or other causes of elevation.</li>
</ul>

**Further Inspection is still required to come to the conclusion, if they are mislabelled or not.**

In [11]:
# Inspecting Direct Bilirubin
direct_bilirubin = df[(df["Direct_Bilirubin"] < 0.0) | (df["Direct_Bilirubin"] > 0.3)]
direct_bilirubin["Dataset"].value_counts()

Dataset
1    228
0     40
Name: count, dtype: int64

There are 40 Records that are having Direct_Bilirubin out side the range of Normal

In [12]:
db_negative = direct_bilirubin[direct_bilirubin["Dataset"]==0]

Inspecting Total_Bilirubin and Direct_Bilirubin records intersect or not.

In [ ]:
# Performing a kind of Inner Join between the two tables to come to the conclusion if the rows intersect or not
intersection_tb_db = pd.merge(tb_negative, db_negative, how="inner")
intersection_tb_db

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,33,Male,1.6,0.5,165,15,23,7.3,3.5,0.92,0
1,38,Female,2.6,1.2,410,59,57,5.6,3.0,0.80,0
2,42,Male,6.8,3.2,630,25,47,6.1,2.3,0.60,0
3,35,Male,1.8,0.6,275,48,178,6.5,3.2,0.90,0
4,70,Male,1.4,0.6,146,12,24,6.2,3.8,1.58,0
5,36,Male,5.3,2.3,145,32,92,5.1,2.6,1.00,0
6,50,Male,5.8,3.0,661,181,285,5.7,2.3,0.67,0
7,50,Male,7.3,3.6,1580,88,64,5.6,2.3,0.60,0
8,58,Male,1.7,0.8,188,60,84,5.9,3.5,1.40,0
9,60,Male,1.8,0.5,201,45,25,3.9,1.7,0.70,0


In [14]:
intersection_tb_db.shape

(33, 11)

<ul>
    <li>These 33 records are consistently abnormal in both features but still labeled 0 → likely borderline or subclinical cases, not mislabels.</li>
    <li>Since both indicators flag the same cases, they're likely highly correlated (as expected physiologically).</li>
    <li>Still, further analysis and evidence is required to come to the conclusion that they are correct/misclassified</li>
</ul>

In [15]:
# Inspecting Alkaline Phosphate
alp_abnormal = df[(df["Alkaline_Phosphotase"] < 40) | (df["Alkaline_Phosphotase"] > 130)]
alp_abnormal["Dataset"].value_counts()

Dataset
1    387
0    157
Name: count, dtype: int64

In [16]:
alp_negative = alp_abnormal[alp_abnormal["Dataset"]==0]

In [17]:
# Doing an innerJoin/Intersection between the AlkalinePhosphotise and already created intersection table of TotalBilirubin and DirectBilirubin
intersection_tb_db_apl = pd.merge(alp_negative, intersection_tb_db, how="inner")

In [19]:
intersection_tb_db_apl

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,33,Male,1.6,0.5,165,15,23,7.3,3.5,0.92,0
1,38,Female,2.6,1.2,410,59,57,5.6,3.0,0.80,0
2,42,Male,6.8,3.2,630,25,47,6.1,2.3,0.60,0
3,35,Male,1.8,0.6,275,48,178,6.5,3.2,0.90,0
4,70,Male,1.4,0.6,146,12,24,6.2,3.8,1.58,0
5,36,Male,5.3,2.3,145,32,92,5.1,2.6,1.00,0
6,50,Male,5.8,3.0,661,181,285,5.7,2.3,0.67,0
7,50,Male,7.3,3.6,1580,88,64,5.6,2.3,0.60,0
8,58,Male,1.7,0.8,188,60,84,5.9,3.5,1.40,0
9,60,Male,1.8,0.5,201,45,25,3.9,1.7,0.70,0


In [20]:
intersection_tb_db_apl.shape

(31, 11)


- There are 31 non-diseased records with abnormal Total_Bilirubin, Direct_Bilirubin, and Alkaline_Phosphotase.
- These three features together suggest significant liver-related abnormality, yet they're still labeled as 0 (non-diseased).
- This strengthens the case that:
- These 30 rows are either edge cases, label noise, or reflect undiagnosed/subclinical liver issues.
- Statistically, these are outliers among the “healthy” class and will likely confuse a classifier unless handled carefully.

**Still, I'll inspect one another medical indicator to further concrete the evidence**

In [21]:
# Inspecting Alamine_Aminotransferase
alt_abnormal = df[(df["Alamine_Aminotransferase"] < 7) | (df["Alamine_Aminotransferase"] > 56)]
alt_abnormal["Dataset"].value_counts()

Dataset
1    139
0     18
Name: count, dtype: int64

In [22]:
# Segregating the Negative Cases
alt_abnormal_negative = alt_abnormal[alt_abnormal["Dataset"]==0]

In [23]:
# Performing Intersextion / Inner Join to find common rows
intersection_tb_db_apl_alt = pd.merge(alp_negative, intersection_tb_db_apl, how="inner")

In [24]:
intersection_tb_db_apl_alt

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,33,Male,1.6,0.5,165,15,23,7.3,3.5,0.92,0
1,38,Female,2.6,1.2,410,59,57,5.6,3.0,0.80,0
2,42,Male,6.8,3.2,630,25,47,6.1,2.3,0.60,0
3,35,Male,1.8,0.6,275,48,178,6.5,3.2,0.90,0
4,70,Male,1.4,0.6,146,12,24,6.2,3.8,1.58,0
5,36,Male,5.3,2.3,145,32,92,5.1,2.6,1.00,0
6,50,Male,5.8,3.0,661,181,285,5.7,2.3,0.67,0
7,50,Male,7.3,3.6,1580,88,64,5.6,2.3,0.60,0
8,58,Male,1.7,0.8,188,60,84,5.9,3.5,1.40,0
9,60,Male,1.8,0.5,201,45,25,3.9,1.7,0.70,0


In [25]:
intersection_tb_db_apl_alt.shape

(31, 11)

The Above Final Result concretes the evidence the Rows are Mislabelled as consistently 31 Rows despite having the abnormal values they were classified as Normal/No Liver Disease

In [26]:
pd.merge(alt_abnormal_negative, intersection_tb_db_apl, how="left")

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,64,Male,0.9,0.3,310,61,58,7.0,3.4,0.90,0
1,25,Male,0.6,0.1,183,91,53,5.5,2.3,0.70,0
2,38,Female,2.6,1.2,410,59,57,5.6,3.0,0.80,0
3,27,Male,1.2,0.4,179,63,39,6.1,3.3,1.10,0
4,50,Male,5.8,3.0,661,181,285,5.7,2.3,0.67,0
5,50,Male,7.3,3.6,1580,88,64,5.6,2.3,0.60,0
6,58,Male,1.7,0.8,188,60,84,5.9,3.5,1.40,0
7,38,Male,1.5,0.4,298,60,103,6.0,3.0,1.00,0
8,22,Male,2.7,1.0,160,82,127,5.5,3.1,1.20,0
9,61,Male,1.5,0.6,196,61,85,6.7,3.8,1.30,0


In [27]:
pd.merge(alt_abnormal_negative, intersection_tb_db_apl, how="right")

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,33,Male,1.6,0.5,165,15,23,7.3,3.5,0.92,0
1,38,Female,2.6,1.2,410,59,57,5.6,3.0,0.80,0
2,42,Male,6.8,3.2,630,25,47,6.1,2.3,0.60,0
3,35,Male,1.8,0.6,275,48,178,6.5,3.2,0.90,0
4,70,Male,1.4,0.6,146,12,24,6.2,3.8,1.58,0
5,36,Male,5.3,2.3,145,32,92,5.1,2.6,1.00,0
6,50,Male,5.8,3.0,661,181,285,5.7,2.3,0.67,0
7,50,Male,7.3,3.6,1580,88,64,5.6,2.3,0.60,0
8,58,Male,1.7,0.8,188,60,84,5.9,3.5,1.40,0
9,60,Male,1.8,0.5,201,45,25,3.9,1.7,0.70,0


In [28]:
pd.merge(alt_abnormal_negative, intersection_tb_db_apl, how="inner")

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,38,Female,2.6,1.2,410,59,57,5.6,3.0,0.80,0
1,50,Male,5.8,3.0,661,181,285,5.7,2.3,0.67,0
2,50,Male,7.3,3.6,1580,88,64,5.6,2.3,0.60,0
3,58,Male,1.7,0.8,188,60,84,5.9,3.5,1.40,0
4,38,Male,1.5,0.4,298,60,103,6.0,3.0,1.00,0
5,22,Male,2.7,1.0,160,82,127,5.5,3.1,1.20,0
6,61,Male,1.5,0.6,196,61,85,6.7,3.8,1.30,0
7,38,Male,2.2,1.0,310,119,42,7.9,4.1,1.00,0
8,39,Male,1.6,0.8,230,88,74,8.0,4.0,1.00,0


In [29]:
pd.merge(alt_abnormal_negative, intersection_tb_db_apl, how="outer")

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,4,Male,0.8,0.2,460,152,231,6.5,3.2,0.90,0
1,18,Male,1.3,0.7,316,10,21,6.0,2.1,0.50,0
2,19,Male,1.4,0.8,178,13,26,8.0,4.6,1.30,0
3,22,Male,0.8,0.2,300,57,40,7.9,3.8,0.90,0
4,22,Male,2.7,1.0,160,82,127,5.5,3.1,1.20,0
5,23,Female,2.3,0.8,509,28,44,6.9,2.9,0.70,0
6,24,Male,3.3,1.6,174,11,33,7.6,3.9,1.00,0
7,25,Male,0.6,0.1,183,91,53,5.5,2.3,0.70,0
8,26,Male,1.9,0.8,180,22,19,8.2,4.1,1.00,0
9,27,Male,1.2,0.4,179,63,39,6.1,3.3,1.10,0


<ul>
    <li>
        <b>Interpretation:</b>
        Now we have a solid cohort of non-diseased records that violate multiple independent liver function indicators.
    </li>
    <li>
        This isn't explainable by natural variation alone — they're either:

Subclinical cases (not labeled yet)

Mislabels

A limitation of the diagnostic label itself.
Even if medically correct, these rows are statistical anomalies in the Dataset = 0 class — and they will likely reduce your classifier’s confidence.
    </li>
</ul>



In [30]:
# Using all columns to match, assuming columns match exactly
df_clean = df.merge(intersection_tb_db_apl, how='outer', indicator=True)
df_clean = df_clean[df_clean['_merge'] == 'left_only'].drop(columns=['_merge'])

In [31]:
df_clean.shape

(539, 11)

In [32]:
df_clean["Dataset"].value_counts()

Dataset
1    406
0    133
Name: count, dtype: int64

In [33]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [34]:
df_clean["Gender"] = df_clean["Gender"].map({"Male":1, "Female":0})

In [35]:

X = df_clean.drop(columns=["Dataset"])
y = df_clean["Dataset"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

In [36]:
# Train with class weights
clf = RandomForestClassifier(class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [38]:
y_pred = clf.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[ 7 20]
 [10 71]]
              precision    recall  f1-score   support

           0      0.412     0.259     0.318        27
           1      0.780     0.877     0.826        81

    accuracy                          0.722       108
   macro avg      0.596     0.568     0.572       108
weighted avg      0.688     0.722     0.699       108



In [39]:
df_clean.isnull().sum()

Age                           0
Gender                        0
Total_Bilirubin               0
Direct_Bilirubin              0
Alkaline_Phosphotase          0
Alamine_Aminotransferase      0
Aspartate_Aminotransferase    0
Total_Protiens                0
Albumin                       0
Albumin_and_Globulin_Ratio    4
Dataset                       0
dtype: int64

In [40]:
df_clean[df_clean["Albumin_and_Globulin_Ratio"].isnull()]

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
88,27,1,1.3,0.6,106,25,54,8.5,4.8,NaN,0
162,35,0,0.6,0.2,180,12,15,5.2,2.7,NaN,0
270,45,0,0.9,0.3,189,23,33,6.6,3.9,NaN,1
369,51,1,0.8,0.2,230,24,46,6.5,3.1,NaN,1


In [41]:
# Filling NaN in Albumin_and_Globulin_Ratio with class-wise median
df_clean["Albumin_and_Globulin_Ratio"] = df_clean.groupby("Dataset")["Albumin_and_Globulin_Ratio"]\
    .transform(lambda x: x.fillna(x.median()))

In [44]:
df_clean.isnull().sum()

Age                           0
Gender                        0
Total_Bilirubin               0
Direct_Bilirubin              0
Alkaline_Phosphotase          0
Alamine_Aminotransferase      0
Aspartate_Aminotransferase    0
Total_Protiens                0
Albumin                       0
Albumin_and_Globulin_Ratio    0
Dataset                       0
dtype: int64

In [48]:
df_clean.head()

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,4,1,0.8,0.2,460,152,231,6.5,3.2,0.9,0
1,4,1,0.9,0.2,348,30,34,8.0,4.0,1.0,0
2,6,1,0.6,0.1,289,38,30,4.8,2.0,0.7,0
3,7,0,27.2,11.8,1420,790,1050,6.1,2.0,0.4,1
4,7,1,0.5,0.1,352,28,51,7.9,4.2,1.1,0


In [49]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [50]:
from sklearn.feature_selection import RFE

In [51]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


rfe = RFE(
    estimator=rf,
    n_features_to_select=6
)


X_train_rfe = rfe.fit_transform(X_train_scaled, y_train)
X_test_rfe = rfe.transform(X_test_scaled)


selected_features = X.columns[rfe.support_]
print('Selected features:', list(selected_features))

Selected features: ['Age', 'Total_Bilirubin', 'Alkaline_Phosphotase', 'Alamine_Aminotransferase', 'Aspartate_Aminotransferase', 'Albumin']


In [52]:
from imblearn.combine import SMOTEENN

In [ ]:
# Applying SMOTEENN to training data
from collections import Counter
print("Before SMOTEENN:", Counter(y_train))

smote_enn = SMOTEENN(random_state=42)
X_train_bal, y_train_bal = smote_enn.fit_resample(
    X_train_rfe, y_train
)

print("After SMOTEENN:", Counter(y_train_bal))

Before SMOTEENN: Counter({1: 325, 0: 106})
After SMOTEENN: Counter({0: 232, 1: 162})


In [56]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_bal, y_train_bal)

y_pred = clf.predict(X_test_rfe)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[26  1]
 [31 50]]
              precision    recall  f1-score   support

           0      0.456     0.963     0.619        27
           1      0.980     0.617     0.758        81

    accuracy                          0.704       108
   macro avg      0.718     0.790     0.688       108
weighted avg      0.849     0.704     0.723       108



In [58]:
from lightgbm import LGBMClassifier

In [60]:
clf = LGBMClassifier(random_state=42)
clf.fit(X_train_bal, y_train_bal)

y_pred = clf.predict(X_test_rfe)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[25  2]
 [30 51]]
              precision    recall  f1-score   support

           0      0.455     0.926     0.610        27
           1      0.962     0.630     0.761        81

    accuracy                          0.704       108
   macro avg      0.708     0.778     0.685       108
weighted avg      0.835     0.704     0.723       108



C:\Users\praka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Modeling

In [61]:
smote_enn = SMOTEENN(random_state=42)
X_train_bal, y_train_bal = smote_enn.fit_resample(
    X_train_rfe, y_train
)
print("Before SMOTEENN:", Counter(y_train))
print("After SMOTEENN:", Counter(y_train_bal))

Before SMOTEENN: Counter({1: 325, 0: 106})
After SMOTEENN: Counter({0: 232, 1: 162})


In [62]:
from lightgbm import LGBMClassifier

In [63]:
lgbm_model = LGBMClassifier()

In [64]:
lgbm_model.fit(X_train_bal, y_train_bal)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [65]:
# Predict on test set
y_pred = lgbm_model.predict(X_test_rfe)

# Evaluate
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[25  2]
 [30 51]]
              precision    recall  f1-score   support

           0      0.455     0.926     0.610        27
           1      0.962     0.630     0.761        81

    accuracy                          0.704       108
   macro avg      0.708     0.778     0.685       108
weighted avg      0.835     0.704     0.723       108



C:\Users\praka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1
)

In [66]:
lgbm_model.fit(X_train_bal, y_train_bal)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [67]:
y_pred = lgbm_model.predict(X_test_rfe)

C:\Users\praka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [68]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[25  2]
 [30 51]]
              precision    recall  f1-score   support

           0      0.455     0.926     0.610        27
           1      0.962     0.630     0.761        81

    accuracy                          0.704       108
   macro avg      0.708     0.778     0.685       108
weighted avg      0.835     0.704     0.723       108



In [69]:
lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05
)

In [70]:
lgbm_model.fit(X_train_bal, y_train_bal)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [71]:
y_pred = lgbm_model.predict(X_test_rfe)

C:\Users\praka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [72]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[24  3]
 [30 51]]
              precision    recall  f1-score   support

           0      0.444     0.889     0.593        27
           1      0.944     0.630     0.756        81

    accuracy                          0.694       108
   macro avg      0.694     0.759     0.674       108
weighted avg      0.819     0.694     0.715       108



In [73]:
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03
)

In [74]:
lgbm_model.fit(X_train_bal, y_train_bal)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.03
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [75]:
y_pred = lgbm_model.predict(X_test_rfe)

C:\Users\praka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [76]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

[[25  2]
 [30 51]]
              precision    recall  f1-score   support

           0      0.455     0.926     0.610        27
           1      0.962     0.630     0.761        81

    accuracy                          0.704       108
   macro avg      0.708     0.778     0.685       108
weighted avg      0.835     0.704     0.723       108



In [77]:
from sklearn.metrics import accuracy_score

y_test_np = y_test.to_numpy()

lgbm_configs = [
{'n_estimators': 100, 'learning_rate': 0.1},
{'n_estimators': 200, 'learning_rate': 0.05},
{'n_estimators': 300, 'learning_rate': 0.03}
]


results = []


for cfg in lgbm_configs:
    print('Training LightGBM with:', cfg)


model = LGBMClassifier(
    **cfg,
    random_state=42
)


model.fit(X_train_bal, y_train_bal)


preds = model.predict(X_test_rfe)
preds = preds.reshape(-1)


acc = accuracy_score(y_test_np, preds)
print('Accuracy:', acc)
print('-' * 40)


results.append({
'params': cfg,
'accuracy': acc
})

Training LightGBM with: {'n_estimators': 100, 'learning_rate': 0.1}
Training LightGBM with: {'n_estimators': 200, 'learning_rate': 0.05}
Training LightGBM with: {'n_estimators': 300, 'learning_rate': 0.03}
Accuracy: 0.7037037037037037
----------------------------------------


C:\Users\praka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
